<a href="https://colab.research.google.com/github/sri-wahyuni10/Inter-flyrank/blob/main/work/notebooks%20/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# My Lane as an ML Task (Type)

My Lane as an ML Task (Type)This project is classified as Binary Classification and Probability Scoring. Technically, the model will not only output a rigid binary label (0 or 1), but rather predict a continuous value in the form of a probability $P(Y=1\vert{}X)$, which is the chance of a patient being readmitted based on their clinical features ($X$). This probability score is crucial because hospital management needs flexibility to determine the decision threshold. If the hospital bed occupancy rate is high, the threshold can be lowered to make the system more sensitive in detecting vulnerable patients.

# Target or Proxy
Our prediction target is the readmitted variable (1 = Readmitted within 30 days; 0 = Not readmitted).

This is a True Target, not a proxy target, because we directly measure the downstream event the hospital is trying to prevent. This target is free from daily subjective bias because it is bound by a strict time parameter (30 days post-discharge). If a patient readmitted after day 31, they are labeled 0 because clinically the causal factors are usually beyond the control of the hospital's initial discharge care management.

# Success Metric

We cannot use Accuracy because this dataset is imbalanced, with the minority class (label 1) only accounting for 18.80%. If the model predicted all patients as "Safe (0)," the accuracy would still be high at 81.20%, but the model would be completely useless.

Therefore, we define two primary metrics:

*  ** Recall (Sensitivity)** as the Primary Metric: We want to maximize the model's ability to identify patients who are truly at risk of readmission. We tolerate false positives (safe patients predicted as at risk) more than false negatives (critical patients predicted as safe).

*   **AUC-ROC as a Supporting Metric**: To measure how well the model can discriminate between high-risk and stable patient groups at various threshold levels.

# The Unit of Analysis, as a Real Dataframe (Code & Sketch)

In designing a Machine Learning system, the Unit of Analysis is the most crucial foundation. The unit of analysis defines what a single row of data in  dataset represents.
1. Real Definition: In this dataset, the unit of analysis is One Patient Discharge Event.
2. Why Not "Per Patient"? Because a single patient (the same Patient_ID) can be admitted and discharged from the hospital multiple times in a year. If we set the unit of analysis as "per patient," we will lose the clinical context of each visit. By setting per discharge event, the ML model can learn the patient's specific condition at that visit (e.g., length of stay or number of medications this week) to predict their risk in the next 30 days.
3. Target Sketch Logic: The target sketch serves to map the future output of the ML loop. The model does not simply guess a rigid label of $0$ or $1$, but calculates a probability score from $0.00$ to $1.00$. This score is what provides value to hospital operations to flexibly filter the severity of a patient's risk.

In [3]:
import pandas as pd
import numpy as np
import kagglehub

In [8]:
# Download latest version
path = kagglehub.dataset_download("vanpatangan/readmission-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'readmission-dataset' dataset.
Path to dataset files: /kaggle/input/readmission-dataset


In [11]:
df = pd.read_csv("/kaggle/input/readmission-dataset/train_df.csv")
df.head()


,age,gender,primary_diagnosis,num_procedures,days_in_hospital,comorbidity_score,discharge_to,readmitted
0,69,Male,Heart Disease,1,2,1,Home Health Care,0
1,32,Female,COPD,2,13,2,Rehabilitation Facility,0
2,89,Male,Diabetes,1,7,1,Home,0
3,78,Male,COPD,9,2,2,Skilled Nursing Facility,0
4,38,Male,Diabetes,6,4,4,Rehabilitation Facility,0


In [14]:
print("="*80)
print("UNIT OF ANALYSIS SPECIFICATION")
print("="*80)
print(f"Each row represents ONE PATIENT DISCHARGE EVENT (Single Discharge Event).")
print(f"Total rows analyzed: {df.shape[0]} baris.")
print(f"Total clinical/demographic features: {df.shape[1] - 2} features (outside of Patient_ID and Target).")
print("-"*80)

# Menampilkan representasi fisik dari Unit of Analysis
display(df.head(3))

print("\n" + "="*80)
print("TARGET COLUMN SKETCH & PROBABILISTIC FRAMEWORK")
print("="*80)

# DETEKSI OTOMATIS KOLOM ID PASIEN
# Mencari kolom yang mengandung kata 'patient' atau 'id' secara case-insensitive
id_col = None
for col in df.columns:
    if 'patient' in col.lower() or 'id' in col.lower():
        id_col = col
        break

# Jika tidak ditemukan sama sekali, gunakan indeks dataframe sebagai pengganti ID
if id_col is None:
    df['Generated_Patient_ID'] = df.index + 1
    id_col = 'Generated_Patient_ID'

# Mengambil sampel 5 baris pertama untuk membuat sketsa target
sketsa_target = df[[id_col, 'readmitted']].head(5).copy()

# Simulasi: Menambahkan kolom probabilitas kontinu yang nantinya akan dihasilkan oleh ML (Skor 0 - 1)
np.random.seed(42) # Menjaga agar angka acak tetap konsisten saat dijalankan ulang
sketsa_target['Predicted_Probability_ML'] = np.round(np.random.uniform(0.05, 0.95, size=5), 4)

# Simulasi: Menentukan keputusan intervensi medis berdasarkan batasan (Threshold) > 0.50
sketsa_target['Final_Decision_Label'] = (sketsa_target['Predicted_Probability_ML'] > 0.5).astype(int)

# Menampilkan tabel simulasi keluaran ML
display(sketsa_target)

print("-"*80)
print("Output Explanation:")
print(f"- '{id_col}': Unique identification to track patient medical events.")
print("- 'readmitted': Actual labels from historical data (Ground Truth).")
print("- 'Predicted_Probability_ML': Results of prediction of patient vulnerability level by ML.")
print("- 'Final_Decision_Label': Final decision. If the probability is > 50%, the patient is marked as requiring intervention..")

UNIT OF ANALYSIS SPECIFICATION
Each row represents ONE PATIENT DISCHARGE EVENT (Single Discharge Event).
Total rows analyzed: 5000 baris.
Total clinical/demographic features: 6 features (outside of Patient_ID and Target).
--------------------------------------------------------------------------------


,age,gender,primary_diagnosis,num_procedures,days_in_hospital,comorbidity_score,discharge_to,readmitted
0,69,Male,Heart Disease,1,2,1,Home Health Care,0
1,32,Female,COPD,2,13,2,Rehabilitation Facility,0
2,89,Male,Diabetes,1,7,1,Home,0



TARGET COLUMN SKETCH & PROBABILISTIC FRAMEWORK


,comorbidity_score,readmitted,Predicted_Probability_ML,Final_Decision_Label
0,1,0,0.3871,0
1,2,0,0.9056,1
2,1,0,0.7088,1
3,2,0,0.5888,1
4,4,0,0.1904,0


--------------------------------------------------------------------------------
Output Explanation:
- 'comorbidity_score': Unique identification to track patient medical events.
- 'readmitted': Actual labels from historical data (Ground Truth).
- 'Predicted_Probability_ML': Results of prediction of patient vulnerability level by ML.
- 'Final_Decision_Label': Final decision. If the probability is > 50%, the patient is marked as requiring intervention..


# Why ML Beats a Fixed Rule Here (In-Depth Analysis)

If we only use fixed rules (or If-Else Statements) as conventional medical teams typically do, prevention accuracy will be very low due to the "Descriptive Homogeneous Pattern" phenomenon we found in last week's data:


*   feature Homogeneity: The average length of stay for readmitted patients was 7.31 days, while the average for those who were safe was 7.41 days. The average comorbidity for both was 2.06. The average age range for both was also identical at around 53 years.
*   Why Fixed Rules Fail: If you create a fixed rule like: IF days_in_hospital > 7 AND comorbidity_score > 2 THEN High_Risk, this rule will fail miserably because both groups have the same mean value. Human rules are insensitive to small variations.


*   Why ML Wins: Machine Learning doesn't look at these variables in isolation. ML algorithms (such as Gradient Boosting or Random Forest) work by mapping high-level nonlinear interactions. ML can detect subtle combination patterns that are invisible to humans, for example: "A female patient, even though her comorbidity score is only 2, if she has a certain trend in the number of procedures, her risk jumps to 19.41%." ML can transform 17 subtle data dimensions into a single, accurate and objective probability score.


